In [1]:
import json
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/processed")

TRAIN_IMAGES = DATA_DIR / "train/images"
TRAIN_ANN = DATA_DIR / "train/annotations"

VAL_IMAGES = DATA_DIR / "validation/images"
VAL_ANN = DATA_DIR / "validation/annotations"

YOLO_TRAIN = DATA_DIR / "train/yolo_labels"
YOLO_VAL = DATA_DIR / "validation/yolo_labels"

YOLO_TRAIN.mkdir(exist_ok=True)
YOLO_VAL.mkdir(exist_ok=True)

In [3]:
def convert_to_yolo(json_file, yolo_file, img_width, img_height):
    
    with open(json_file) as f:
        data = json.load(f)
    
    lines = []
    
    for key, item in data.items():
        if key.startswith("item"):
            bbox = item.get("bounding_box")
            class_id = item.get("category_id")
            
            if bbox is None or class_id is None:
                continue
            
            x1, y1, x2, y2 = bbox
            
            # convert to YOLO format
            x_center = ((x1 + x2) / 2) / img_width
            y_center = ((y1 + y2) / 2) / img_height
            w = (x2 - x1) / img_width
            h = (y2 - y1) / img_height
            
            lines.append(f"{class_id} {x_center} {y_center} {w} {h}")
    
    if lines:
        with open(yolo_file, "w") as f:
            f.write("\n".join(lines))


In [4]:
for json_file in list(TRAIN_ANN.glob("*.json"))[:100]:
    
    img_file = TRAIN_IMAGES / f"{json_file.stem}.jpg"
    
    # get image size
    from PIL import Image
    img = Image.open(img_file)
    w, h = img.size
    
    yolo_file = YOLO_TRAIN / f"{json_file.stem}.txt"
    
    convert_to_yolo(json_file, yolo_file, w, h)

print("Sample conversion done for 100 training images.")


Sample conversion done for 100 training images.


In [ ]:
# training folder
for json_file in TRAIN_ANN.glob("*.json"):
    img_file = TRAIN_IMAGES / f"{json_file.stem}.jpg"
    img = Image.open(img_file)
    w, h = img.size
    yolo_file = YOLO_TRAIN / f"{json_file.stem}.txt"
    convert_to_yolo(json_file, yolo_file, w, h)

# validation folder
for json_file in VAL_ANN.glob("*.json"):
    img_file = VAL_IMAGES / f"{json_file.stem}.jpg"
    img = Image.open(img_file)
    w, h = img.size
    yolo_file = YOLO_VAL / f"{json_file.stem}.txt"
    convert_to_yolo(json_file, yolo_file, w, h)

print("full yolo conversion done for train and validation")


Full YOLO conversion done for train and validation.
